In [ ]:
# Network segments to evaluate. Edit these before running.
# Enter any address in the target /24 (e.g. "10.1.5.0" or "10.1.5.0/24").
oco = "X.X.X.0"   # OCO: the subnet YOUR group attacked
dco = "X.X.X.0"   # DCO: YOUR group's subnet


In [ ]:
# -*- coding: utf-8 -* #
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
import traceback
import sys
import datetime
import pandas as pd
import warnings
warnings.filterwarnings("ignore")   # hide Python warnings; errors still show
from IPython.display import display

from df_binning import *
from generate_individual_labels_dataframe import *
import pyspark.sql.functions as F
from pyspark.sql.window import Window

#############
# Run variables
#############

benign_label = "none"

conn_server_loc = '/home/kali/datasets/CNT4416_full_csv'

log_location = "/home/kali/datasets/randomforest/logs/randomForest/" + str(datetime.date.today()) + "_log.txt"

########################
# functions
########################

def segment_prefix(addr):
    """First three octets plus a trailing dot, e.g. '10.1.5.0/24' -> '10.1.5.'"""
    octets = addr.strip().split("/")[0].split(".")
    if len(octets) < 3 or not all(o.isdigit() and 0 <= int(o) <= 255 for o in octets[:3]):
        raise ValueError("'" + addr + "' is not a valid IPv4 network segment")
    return ".".join(octets[:3]) + "."

GAP_SECONDS = 120   # same 2-minute window the correlation script uses

def segment_attack_events(preds, ip_col, prefix, gap_seconds=GAP_SECONDS):
    """Condenses a segment's attack connections into attacks so counts can be compared with
    mission logs. Connections from the same source IP with the same technique (T code) belong
    to one attack until there is a gap of more than gap_seconds between them; a longer gap
    starts a new attack. Rows the correlation script marked "Duplicate" (extra copies made
    when one technique maps to several tactics) are skipped so they are not counted twice.
    An attack counts as detected if the model flagged at least one of its connections.
    Returns (number of attacks, number of detected attacks)."""
    attacks = (preds
               .filter(F.col(ip_col).cast("string").startswith(prefix)
                       & (F.col("label_bin") == 0.0)
                       & (F.coalesce(F.col("label_technique").cast("string"), F.lit("")) != "Duplicate")
                       & F.col("ts").isNotNull())
               .select(F.col("src_ip").cast("string").alias("src"),
                       F.col("label_technique").cast("string").alias("technique"),
                       F.col("ts").cast("double").alias("ts_s"), "prediction"))
    w = Window.partitionBy("src", "technique").orderBy("ts_s")
    attacks = attacks.withColumn("gap", F.col("ts_s") - F.lag("ts_s").over(w))
    attacks = attacks.withColumn("new_attack",
                                 F.when(F.col("gap").isNull() | (F.col("gap") > gap_seconds), 1).otherwise(0))
    attacks = attacks.withColumn("attack_id",
                                 F.sum("new_attack").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    per_attack = attacks.groupBy("src", "technique", "attack_id").agg(
        F.max(F.when(F.col("prediction") == 0.0, 1).otherwise(0)).alias("detected"))
    r = per_attack.agg(F.count(F.lit(1)).alias("n"), F.sum("detected").alias("d")).collect()[0]
    return int(r["n"]), int(r["d"] or 0)

def segment_benign_windows(preds, ip_col, prefix, window_seconds=GAP_SECONDS):
    """Groups a segment's benign connections into fixed 2-minute windows. A window counts
    as a false alarm if the model flagged at least one of its connections as an attack.
    Returns (number of windows, number of flagged windows)."""
    windows = (preds
               .filter(F.col(ip_col).cast("string").startswith(prefix)
                       & (F.col("label_bin") == 1.0)
                       & F.col("ts").isNotNull())
               .withColumn("window", F.floor(F.col("ts").cast("double") / window_seconds))
               .groupBy("window")
               .agg(F.max(F.when(F.col("prediction") == 0.0, 1).otherwise(0)).alias("flagged")))
    r = windows.agg(F.count(F.lit(1)).alias("n"), F.sum("flagged").alias("f")).collect()[0]
    return int(r["n"]), int(r["f"] or 0)

oco_prefix = segment_prefix(oco)
dco_prefix = segment_prefix(dco)
print("OCO segment (dest_ip): " + oco_prefix + "x")
print("DCO segment (dest_ip): " + dco_prefix + "x")

########################
# code start
########################

spark = ( 
    SparkSession
    .builder
    .master("local[*]")
    .appName("TM_RandoForest")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
    )

# Only show Spark errors (hides WARN/INFO messages such as maxToStringFields)
spark.sparkContext.setLogLevel("ERROR")

printToLog("Begin run\n-----------\n----------\n\n", log_location)
printToLog("Dataset location - " + conn_server_loc, log_location)

source_df = spark.read.option("header", True).options(inferSchema=True).csv(conn_server_loc)
 
try:
    source_df = rename_columns(source_df)
except Exception as e:
    print("Error:  " + str(e), file=sys.stderr)
    print(traceback.format_exc())   

# Binary label: any technique label other than "none" = attack (0.0), "none" = benign (1.0)
unbinned_df = source_df.withColumn("label_bin", when((col("label_multi") != benign_label), 0.0).otherwise(1.0))
printToLog(str(unbinned_df.count()) + " rows", log_location)

#Carry out the binning on the df
numeric_percent_trim = 0.02
nominal_percent_agg = 0.98
replace_bool = False

attrList_ip_addr = ['dest_ip','src_ip']
attrList_port = ['dest_port', 'src_port']
attrList_bool = ['local_orig', 'local_resp']
attrList_nominal = ['protocol', 'conn_state', 'history','service']
attrList_numeric = ['duration','orig_bytes','orig_pkts','orig_ip_bytes','resp_bytes','resp_pkts','resp_ip_bytes','missed_bytes']

printToLog("Binning start", log_location)
conn_df = genFullBinnedDF(unbinned_df, attrList_ip_addr, attrList_port, attrList_bool, attrList_nominal, nominal_percent_agg, attrList_numeric, numeric_percent_trim, replace_bool).persist()
printToLog("Binning finished", log_location)

# balanced class weights rarer classes get a higher weight so the
# model is penalized more for missing them. Same formula sklearn uses for
# class_weight='balanced': total / (num_classes * class_count).
class_counts = conn_df.groupBy("label_bin").count()
total_count = conn_df.count()
num_classes = class_counts.count()
class_counts = class_counts.withColumn("class_weight", total_count / (num_classes * col("count")))
conn_df = conn_df.join(class_counts.select("label_bin", "class_weight"), on="label_bin", how="left").persist()

########################
# Build Feature cols
########################     

feature_cols = []
for clm, types in conn_df.dtypes:
    if "_bin" in clm:
        if not "label" in clm:
            feature_cols.append(clm)

# build feature vector for Classifier
assembler = VectorAssembler(inputCols = feature_cols, outputCol = "features")
conn_df = assembler.transform(conn_df)

printToLog("feature_cols - " + str(feature_cols), log_location)
print(str(len(feature_cols)) + " features: " + str(feature_cols))

########################
# Model: trained on every connection EXCEPT those going to the OCO and DCO subnets,
# so the model is scored on traffic it has never seen
########################   

in_segments = coalesce(col("dest_ip").cast("string").startswith(oco_prefix) |
                       col("dest_ip").cast("string").startswith(dco_prefix), lit(False))
train_df = conn_df.filter(~in_segments)
score_df = conn_df.filter(in_segments)

rf = RandomForestClassifier(featuresCol = "features", labelCol = "label_bin", weightCol = "class_weight")
rfModel = rf.fit(train_df)
printToLog("randomForest model fit", log_location)

#############
# Predictions on the OCO and DCO connections only
#############   

predictions = rfModel.transform(score_df)

for ip_c in ("dest_ip", "src_ip", "ts", "label_technique"):
    if ip_c not in predictions.columns:
        raise RuntimeError("'" + ip_c + "' is not in the dataframe; the OCO/DCO counts need it")

################################
# Output: confusion matrices counting only connections destined to the OCO and DCO subnets
################################

segments = [("OCO", "dest_ip", oco_prefix), ("DCO", "dest_ip", dco_prefix)]

# Event-level confusion matrix, same table format as TM_ML.randForestMaster.
# Positive = attack. Connections are grouped so counts can be compared with mission logs:
#   TP / FN: attacks (same source IP and T code, split at gaps over 2 minutes) detected / missed
#   FP / TN: benign 2-minute windows the model flagged / did not flag
for seg_name, ip_col, prefix in segments:
    n_conn = predictions.filter(col(ip_col).cast("string").startswith(prefix)).count()
    n_events, tp = segment_attack_events(predictions, ip_col, prefix)
    n_windows, fp = segment_benign_windows(predictions, ip_col, prefix)
    fn = n_events - tp
    tn = n_windows - fp

    two_by_two = pd.DataFrame(
        [[f"TP: {tp}", f"FN: {fn}"],
         [f"FP: {fp}", f"TN: {tn}"]],
        index=["Actual: Attack", "Actual: Benign"],
        columns=["Predicted: Attack", "Predicted: Benign"]
    )

    print(f"\n{seg_name}:")
    print(f"  {ip_col} {prefix}x  |  {n_conn:,} connections")
    print(f"  Attacks: {n_events:,} (same source IP and T code, split at gaps over 2 minutes)")
    print(f"  Benign:  {n_windows:,} two-minute windows")
    display(two_by_two)

    # Same formulas as TM_ML.randForestMaster, with attack as the positive class
    total = tp + fn + fp + tn
    accuracy = (tp + tn) / total if total > 0 else float("nan")
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else float("nan"))
    print(f"  Accuracy: {accuracy:.1%}   "
          f"Precision: {precision:.1%}   "
          f"Recall: {recall:.1%}   "
          f"F1: {f1:.1%}")
    if n_conn == 0:
        print(f"  No connections matched {prefix}x - check the {seg_name} address.")

conn_df.unpersist()
printToLog("End run\n-----------\n----------\n\n", log_location)
